# Ejercicio 1:

Usando la API de Reddit (o PRAW), autentica y extrae los 5 posts más populares del subreddit "datascience".

Almacena el título y el contenido en un DataFrame de Pandas.

In [ ]:
# Solución Ejercicio 1: API de Reddit con PRAW
import praw
import pandas as pd

reddit = praw.Reddit(client_id="TU_CLIENT_ID",
                     client_secret="TU_CLIENT_SECRET",
                     user_agent="TU_APP")
subreddit = reddit.subreddit("datascience")
posts = []
for post in subreddit.hot(limit=5):
    posts.append({"title": post.title, "content": post.selftext})
df_reddit = pd.DataFrame(posts)
print(df_reddit)


# Ejercicio 2:

Realiza web scraping de la página https://www.parascrapear.com para extraer:
- Los títulos de los posts
- Los nombres de los autores

Guarda los resultados en un CSV.

In [ ]:
# Solución Ejercicio 2: Web Scraping con BeautifulSoup
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://www.parascrapear.com"
res = requests.get(url)
soup = BeautifulSoup(res.text, 'html.parser')
titulos = [h2.text.strip() for h2 in soup.select("section.blog-entry h2")]
autores = [span.text.strip() for span in soup.select("section.blog-entry span.author")]
df_scraping = pd.DataFrame({"title": titulos, "author": autores})
df_scraping.to_csv("scraped_posts.csv", index=False)
print(df_scraping.head())

# Ejercicio 3:

A partir de un archivo JSON (simulado o real) con tweets, utiliza Pandas para:
- Contar el número de tweets por usuario
- Extraer hashtags usando expresiones regulares

Guardar el conteo en otro CSV.

In [ ]:
# Solución Ejercicio 3: Transformación de tweets con Pandas
import pandas as pd
import re
from collections import Counter

df_tweets = pd.read_json("tweets_datascience.json")
# Conteo de tweets por usuario
tweets_count = df_tweets.groupby("user").agg({"content": "count"}).rename(columns={"content": "num_tweets"})
tweets_count.to_csv("tweets_count.csv")
# Extracción de hashtags
df_tweets["hashtags"] = df_tweets["content"].str.findall(r"#\w+")
df_exploded = df_tweets.explode("hashtags").dropna(subset=["hashtags"])
hashtags_count = df_exploded.groupby("hashtags").size().reset_index(name="count").sort_values("count", ascending=False)
hashtags_count.to_csv("hashtags_count.csv", index=False)
print(hashtags_count.head())

# Ejercicio 4:

Con PySpark, lee un archivo JSON de tweets y realiza una transformación:
- Filtra tweets que contengan el hashtag "#datascience"
- Agrupa por usuario y cuenta los tweets
- Escribe el resultado en formato CSV.

In [ ]:
# Solución Ejercicio 4: Transformación con PySpark
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Ejercicio_PySpark").master("local[*]").getOrCreate()
df_spark = spark.read.json("tweets_datascience.json")
df_filtered = df_spark.filter(df_spark.content.contains("#datascience"))
df_grouped = (df_filtered.groupBy("user")
                           .count()
                           .orderBy("count", ascending=False))
df_grouped.show()
df_grouped.write.mode("overwrite").csv("tweets_transformed_pyspark")
spark.stop()